# Build Static Feature Grid

**Purpose:** Join all time-invariant spatial attributes onto a single
`(lon, lat)` lookup table. This static grid is reused by:

- `03_02` — adds OSM density features on top of it
- `03_04` — joins it onto the monthly model dataset

| Source file | Columns added |
|-------------|---------------|
| `lon_lat_pair_weather_match_veg_v2.parquet` | `fire_attribute`, `veg` |
| `lon_lat_pair_weather_match_slope.parquet` | `slope_avg`, `slope_max` |
| `road_density_match_weather_grid.parquet` | `road_density_km_km2` |
| `Subregion_Data/lon_lat_pair_weather_match_subregion.parquet` | `SubRegion` |

**Output:** `Clean_Data/static_features.parquet`

> **Run order:** This notebook (`02_05`) must run **before `03_02`**, which reads
> `static_features.parquet` as its base grid.

## 0. Configuration

Centralized path configuration — **edit this cell only**.

In [1]:
import os

# ====================== EDIT THESE PATHS ======================
PROJECT_ROOT = r"E:\zcao\CA_Wildfire"

# Input paths
VEG_PATH       = os.path.join(PROJECT_ROOT, "Clean_Data", "Veg_Data",
                              "lon_lat_pair_weather_match_veg_v2.parquet")
SLOPE_PATH     = os.path.join(PROJECT_ROOT, "Clean_Data", "Slope_Data",
                              "lon_lat_pair_weather_match_slope.parquet")
ROAD_PATH      = os.path.join(PROJECT_ROOT, "Clean_Data", "Road_Data",
                              "road_density_match_weather_grid.parquet")
SUBREGION_PATH = os.path.join(PROJECT_ROOT, "Clean_Data", "Subregion_Data",
                              "lon_lat_pair_weather_match_subregion.parquet")

# Output
OUTPUT_PATH = os.path.join(PROJECT_ROOT, "Clean_Data", "static_features.parquet")

print(f"Output : {OUTPUT_PATH}")
for label, path in [
    ('VEG',       VEG_PATH),
    ('SLOPE',     SLOPE_PATH),
    ('ROAD',      ROAD_PATH),
    ('SUBREGION', SUBREGION_PATH),
]:
    print(f"  [{'OK' if os.path.exists(path) else 'MISSING'}] {label}")

Output : E:\zcao\CA_Wildfire\Clean_Data\static_features.parquet
  [OK] VEG
  [OK] SLOPE
  [OK] ROAD
  [OK] SUBREGION


## 1. Environment Setup

In [2]:
import sys, gc, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

gc.collect()
print(f"Python : {sys.version.split('|')[0].strip()}")
print(f"pandas : {pd.__version__}")

Python : 3.9.13 (main, Aug 25 2022, 23:51:50) [MSC v.1916 64 bit (AMD64)]
pandas : 2.2.2


---

## 2. Load Source Tables

Load each static spatial dataset and trim to only the columns needed.

In [6]:
veg_data       = pd.read_parquet(VEG_PATH)[['lon','lat','fire_attribute','veg']]
slope_data     = pd.read_parquet(SLOPE_PATH)[['lon','lat','slope_avg','slope_max']]
road_data      = pd.read_parquet(ROAD_PATH)[['lon','lat','road_density_km_km2']]
subregion_data = pd.read_parquet(SUBREGION_PATH)

## 3. Join onto Static Grid

Start from the subregion table (defines the final spatial domain after
veg + subregion filtering). Inner-join each source table on `(lon, lat)`.
Assert unique keys before each join to catch any data issues early.

In [7]:
static_features = subregion_data[['lon','lat']].drop_duplicates()
print(f"Starting grid : {static_features.shape[0]:,} cells")

data_list = [('veg',       veg_data),
             ('slope',     slope_data),
             ('road',      road_data),
             ('subregion', subregion_data)]

for name, df in data_list:
    n_unique = df[['lon','lat']].drop_duplicates().shape[0]
    if n_unique != df.shape[0]:
        raise ValueError(f"{name}: non-unique (lon,lat) pairs "
                         f"({df.shape[0]:,} rows, {n_unique:,} unique)")
    before = static_features.shape[0]
    static_features = static_features.merge(df, on=['lon','lat'], how='inner')
    print(f"  {name:<12} : {before:,} → {static_features.shape[0]:,} rows | "
          f"{static_features.shape[1]} cols")

print(f"\nFinal shape : {static_features.shape}")
print(f"Columns     : {list(static_features.columns)}")

Starting grid : 13,048 cells
  veg          : 13,048 → 13,048 rows | 4 cols
  slope        : 13,048 → 13,048 rows | 6 cols
  road         : 13,048 → 13,048 rows | 7 cols
  subregion    : 13,048 → 13,048 rows | 8 cols

Final shape : (13048, 8)
Columns     : ['lon', 'lat', 'fire_attribute', 'veg', 'slope_avg', 'slope_max', 'road_density_km_km2', 'SubRegion']


## 4. Save Output

In [8]:
static_features.to_parquet(OUTPUT_PATH, index=False)

print(f"Saved -> {OUTPUT_PATH}")
print(f"File size  : {os.path.getsize(OUTPUT_PATH)/1e6:.1f} MB")
print(f"Final shape: {static_features.shape[0]:,} rows × {static_features.shape[1]} cols")

del veg_data, slope_data, road_data, subregion_data
gc.collect()

Saved -> E:\zcao\CA_Wildfire\Clean_Data\static_features.parquet
File size  : 0.2 MB
Final shape: 13,048 rows × 8 cols


0

## 5. Summary

| Step | Source | Columns added |
|------|--------|---------------|
| Base grid | Subregion (defines domain) | `lon`, `lat` |
| + Vegetation | `lon_lat_pair_weather_match_veg_v2.parquet` | `fire_attribute`, `veg` |
| + Slope | `lon_lat_pair_weather_match_slope.parquet` | `slope_avg`, `slope_max` |
| + Road density | `road_density_match_weather_grid.parquet` | `road_density_km_km2` |
| + Subregion | `lon_lat_pair_weather_match_subregion.parquet` | `SubRegion` |
| Save | `static_features.parquet` | Input for `03_02` and `03_04` |